# Nhánh 1 — So sánh kiến trúc (Day 1 experiment)

So sánh 4 candidate cho bộ phân loại SQLi đa lớp, chọn theo **F1-macro vs latency vs size** (không mặc định chọn transformer).

Dữ liệu: `data/processed/nhanh1_train.csv` (68.159 dòng, 6 lớp, train/test 54.527/13.632).
Kết quả số liệu đọc từ `reports/nhanh1_architecture_comparison.json` (sinh bởi `scripts/compare_nhanh1_architectures.py`).


In [ ]:
import json
import pandas as pd

with open('../reports/nhanh1_architecture_comparison.json') as f:
    results = json.load(f)

rows = []
for name, m in results.items():
    rows.append({
        'model': name,
        'F1_macro': round(m['f1_macro'], 4),
        'p50_ms': round(m['latency']['p50_ms'], 3),
        'p95_ms': round(m['latency']['p95_ms'], 3),
        'train_s': round(m['train_time_s'], 1),
        'size_MB': round(m['model_size_bytes']/1024/1024, 2),
    })
df = pd.DataFrame(rows)
df

## Per-class F1

In [ ]:
classes = ['normal','union_based','error_based','boolean_blind','time_blind','stacked']
per_class = {name: {c: round(m['per_class'][c]['f1-score'], 3) for c in classes} for name, m in results.items()}
pd.DataFrame(per_class).T

## Nhận xét & Quyết định

| Model | F1-macro | p50 latency | Size | Ghi chú |
|---|---|---|---|---|
| TF-IDF + LogReg | 0.9849 | 0.51ms | 3.9MB | Nhanh, đơn giản |
| TF-IDF + LightGBM | 0.9927 | 60.0ms | 6.0MB | F1 cao nhất **nhưng latency ~60ms** (chậm gấp ~120x) |
| DistilBERT | 0.9919 | 2.79ms | 256MB | F1 cao, latency ổn (GPU), **size lớn + train 24 phút** |
| CNN + SQL-tokenizer | 0.9871 | 0.293ms | 116KB | **Nhanh nhất, nhỏ nhất (28K params)**, F1 sát nhóm dẫn đầu |

**Quyết định: chọn TF-IDF + LogReg** làm baseline production cho Nhánh 1.
- Chênh lệch F1 giữa 4 model rất nhỏ (0.985–0.993) → không đáng đánh đổi.
- LightGBM tuy F1 cao nhất nhưng latency 60ms là quá cao cho database proxy real-time.
- DistilBERT không cho lợi ích F1 rõ rệt mà tốn 256MB + cần GPU + train lâu.
- CNN là ứng viên thay thế tốt (nhanh/nhỏ nhất) nếu cần thêm khả năng học đặc trưng.

⚠️ **Cảnh báo quan trọng:** F1 cao đồng loạt (~0.99) và lớp `stacked` đạt **100% recall ở cả 4 model** dù chỉ có 363 mẫu synthetic → dấu hiệu dữ liệu **quá dễ phân biệt**, không phản ánh độ khó thật của tấn công che giấu (obfuscated). Con số F1 này **không nên hiểu là hệ thống đã "gần hoàn hảo"** — tập test adversarial (Ngày 7) mới là thước đo thật.
